# Collections Analytics — Golden Dataset & Investigation

## What I am trying to answer

The business headline says recovery improved by about 11% month-on-month. Before accepting that number, I want to answer four questions: **what changed, could data quality or portfolio mix explain it, how much can I attribute to collection activity, and what would I need to prove before spending the next ₹10 Cr?**

I am keeping the analysis transparent and practical. Where the data does not support a strong conclusion, I say so.


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

# Keep the notebook portable: it works from either the project root or notebook/.
roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(p for p in roots if (p / "data" / "raw").exists())
RAW = PROJECT_ROOT / "data" / "raw"
GOLDEN = PROJECT_ROOT / "data" / "golden"

print("Project:", PROJECT_ROOT)
print("Raw files:", len(list(RAW.glob("*.csv"))))


Project: /Users/kartikkhati/Desktop/collections_assignment_ready
Raw files: 18


## 1. First look at the raw files

I started with a simple inventory. The main thing I wanted to check was whether row counts could be treated as entity counts. They cannot: several datasets contain repeated IDs or repeated events.


In [14]:
inventory=[]
for f in sorted(RAW.glob("*.csv")):
    df=pd.read_csv(f)
    inventory.append({"file":f.name,"rows":len(df),"columns":len(df.columns),"exact_duplicates":int(df.duplicated().sum())})
inventory_df=pd.DataFrame(inventory).sort_values("rows",ascending=False)
display(inventory_df)


,file,rows,columns,exact_duplicates
5,call_attempts.csv,120000,9,0
7,calls.csv,91350,11,1271
17,whatsapp_events.csv,60600,8,600
0,account_status_history.csv,60000,8,0
10,daily_targeting.csv,45000,7,0
15,sms_events.csv,45000,8,0
6,call_dispositions.csv,35000,8,0
4,borrowers.csv,30600,8,600
3,agents.csv,30000,8,0
1,accounts.csv,30000,11,0


## 2. Primary-key checks

I checked the main business IDs before doing any joins. This is where I found that borrowers, agents, calls and payments contain repeated IDs.


In [15]:
keys=[("borrowers.csv","borrower_id"),("accounts.csv","account_id"),("agents.csv","agent_id"),("calls.csv","call_id"),("payments.csv","payment_id")]
rows=[]
for file_name,key in keys:
    df=pd.read_csv(RAW/file_name)
    rows.append({"entity":file_name[:-4],"raw_rows":len(df),"unique_ids":int(df[key].nunique()),"null_ids":int(df[key].isna().sum())})
display(pd.DataFrame(rows))


,entity,raw_rows,unique_ids,null_ids
0,borrowers,30600,11015,0
1,accounts,30000,30000,0
2,agents,30000,1000,0
3,calls,91350,90000,0
4,payments,25500,25000,0


## 3. Borrower identity conflicts

The same borrower ID can appear with different profile attributes. I do not want to solve this by deleting every conflict, so the Golden Dataset keeps one representative profile and an explicit conflict flag.


In [16]:
b=pd.read_csv(RAW/"borrowers.csv",parse_dates=["created_at","updated_at"])
conf=(b.groupby("borrower_id").agg(names=("name","nunique"),phones=("phone","nunique"),emails=("email","nunique"),cities=("city","nunique"),states=("state","nunique")).reset_index())
conf["identity_conflict"]=conf[["names","phones","emails","cities","states"]].gt(1).any(axis=1)
print("Raw borrower rows:",len(b))
print("Unique borrower IDs:",b["borrower_id"].nunique())
print("Exact duplicate rows:",int(b.duplicated().sum()))
print("IDs with attribute conflicts:",int(conf.identity_conflict.sum()))

golden_borrowers=b.sort_values(["borrower_id","updated_at"]).drop_duplicates("borrower_id",keep="last").copy()
golden_borrowers["identity_conflict_flag"]=golden_borrowers.borrower_id.isin(set(conf.loc[conf.identity_conflict,"borrower_id"]))
print("Golden borrowers:",len(golden_borrowers))


Raw borrower rows: 30600
Unique borrower IDs: 11015
Exact duplicate rows: 600
IDs with attribute conflicts: 8518
Golden borrowers: 11015


## 4. Payment deduplication — the check that matters most

Duplicate financial events can directly inflate recovery, so I checked payments before calculating any headline recovery number.


In [17]:
pmt=pd.read_csv(RAW/"payments.csv",parse_dates=["event_at"])
raw_success=pmt.loc[pmt.payment_status.eq("SUCCESS") & pmt.amount.gt(0),"amount"].sum()
pmt_clean=pmt.drop_duplicates().copy()
golden_success=pmt_clean.loc[pmt_clean.payment_status.eq("SUCCESS") & pmt_clean.amount.gt(0),"amount"].sum()
print("Raw payment rows:",len(pmt))
print("Unique payment IDs:",pmt.payment_id.nunique())
print("Exact duplicate rows:",int(pmt.duplicated().sum()))
print(f"Raw successful recovery: ₹{raw_success/1e7:.2f} Cr")
print(f"After exact-dedup: ₹{golden_success/1e7:.2f} Cr")
print(f"Reduction: ₹{(raw_success-golden_success)/1e7:.2f} Cr")


Raw payment rows: 25500
Unique payment IDs: 25000
Exact duplicate rows: 486
Raw successful recovery: ₹134.15 Cr
After exact-dedup: ₹131.65 Cr
Reduction: ₹2.50 Cr


### What this means

The payment data is not safe to use directly for executive reporting. The final Golden payment data used in the project is about **₹131.56 Cr**, compared with about **₹134.15 Cr** in the raw successful amount — roughly **₹2.59 Cr** difference.

That was enough to convince me that canonicalization needed to happen before the recovery analysis.


## 5. Calls and missing agent IDs

Calls are event data, so exact duplicate events should be removed. I kept calls with missing agent IDs because the interaction itself is still useful for account-level analysis.


In [18]:
calls=pd.read_csv(RAW/"calls.csv",parse_dates=["event_at"])
print("Raw calls:",len(calls))
print("Unique call IDs:",calls.call_id.nunique())
print("Exact duplicate rows:",int(calls.duplicated().sum()))
print("Missing agent IDs:",int(calls.agent_id.isna().sum()))
golden_calls=calls.drop_duplicates().copy()
print("Golden call rows:",len(golden_calls))


Raw calls: 91350
Unique call IDs: 90000
Exact duplicate rows: 1271
Missing agent IDs: 1827
Golden call rows: 90079


## 6. Missingness and relationship checks

I did not fill missing values with guesses. For relationships, I checked whether the join was actually supported by another table.


In [19]:
missing=[]
for fn,cols in {"accounts.csv":["borrower_id"],"borrowers.csv":["email","phone"],"call_attempts.csv":["vendor_id"],"calls.csv":["agent_id"],"field_visits.csv":["scheduled_at"],"payments.csv":["payment_reference"]}.items():
    df=pd.read_csv(RAW/fn)
    for c in cols:
        n=int(df[c].isna().sum()); missing.append({"file":fn,"field":c,"missing":n,"missing_pct":round(100*n/len(df),2)})
display(pd.DataFrame(missing))

accounts=pd.read_csv(RAW/"accounts.csv")
b_ids=set(golden_borrowers.borrower_id)
missing_b=int(accounts.borrower_id.isna().sum())
unresolved=int((accounts.borrower_id.notna() & ~accounts.borrower_id.isin(b_ids)).sum())
print("Missing account→borrower:",missing_b)
print("Unresolved account→borrower:",unresolved)
print("Total unresolved relationship:",missing_b+unresolved)


,file,field,missing,missing_pct
0,accounts.csv,borrower_id,455,1.52
1,borrowers.csv,email,895,2.92
2,borrowers.csv,phone,614,2.01
3,call_attempts.csv,vendor_id,2400,2.00
4,calls.csv,agent_id,1827,2.00
5,field_visits.csv,scheduled_at,250,1.00
6,payments.csv,payment_reference,382,1.50


Missing account→borrower: 455
Unresolved account→borrower: 2458
Total unresolved relationship: 2913


## 7. Timezone and status-history checks

The operational data contains multiple timezones, so I preserve the source timezone and use UTC for cross-region sequencing. I also treat account status history as an event table rather than a single current-state table.


In [20]:
print("Call timezones:")
display(calls.timezone.value_counts(dropna=False))
print("Missing call timestamps:",int(calls.event_at.isna().sum()))

status=pd.read_csv(RAW/"account_status_history.csv",parse_dates=["event_at","recorded_at"])
print("Status history rows:",len(status))
print("Accounts represented:",status.account_id.nunique())
print("Accounts with multiple status events:",int((status.groupby("account_id").size()>1).sum()))
display(status.status.value_counts())


Call timezones:


timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64

Missing call timestamps: 0
Status history rows: 60000
Accounts represented: 25999
Accounts with multiple status events: 17821


status
PAID          8650
CLOSED        8614
DELINQUENT    8612
NPA           8612
WRITEOFF      8583
ACTIVE        8518
PTP           8411
Name: count, dtype: int64

## 8. Recovery — the denominator matters

A fixed 30,000-account denominator would not be appropriate for every month because the eligible portfolio changes. In the main project analysis I therefore compare recovery with the eligible portfolio and also use recovery per eligible account.

The key result is: **gross recovery increased 11.03% from February to March, while recovery per eligible account fell 18.37%.**


## 9. Portfolio mix, selection and survivorship

Risk segments and loan types are broadly balanced, so there is no obvious large mix shift explaining March. I also checked how many accounts were actually reached by collection activity.

**Observed population:** 30,000 accounts

**Ever called:** 28,408

**Never called:** 1,592

**Ever attempted:** 29,451

**Never attempted:** 549

This is why I would not treat “called” versus “not called” as a causal treatment/control split.


In [21]:
risk_mix=(accounts.risk_segment.value_counts(normalize=True)*100).round(2).rename("share_pct").reset_index(); risk_mix.columns=["risk_segment","share_pct"]
loan_mix=(accounts.loan_type.value_counts(normalize=True)*100).round(2).rename("share_pct").reset_index(); loan_mix.columns=["loan_type","share_pct"]
print("Risk mix"); display(risk_mix)
print("Loan mix"); display(loan_mix)

called=set(calls.account_id.dropna()); attempts=pd.read_csv(RAW/"call_attempts.csv"); attempted=set(attempts.account_id.dropna())
print("Ever called:",len(called),"Never called:",accounts.account_id.nunique()-len(called))
print("Ever attempted:",len(attempted),"Never attempted:",accounts.account_id.nunique()-len(attempted))


Risk mix


,risk_segment,share_pct
0,HIGH,25.17
1,MEDIUM,25.11
2,LOW,25.04
3,NPA,24.67


Loan mix


,loan_type,share_pct
0,CREDIT_CARD,20.27
1,AUTO,20.26
2,PERSONAL,19.94
3,CONSUMER,19.77
4,BNPL,19.76


Ever called: 28408 Never called: 1592
Ever attempted: 29451 Never attempted: 549


## 10. Attribution check

A payment can follow several interactions, so latest-touch is useful for description but not proof of causality. I use 7-day, 14-day and 30-day windows in the broader analysis.


In [22]:
latest_touch=pd.DataFrame({"latest_touch":["NO_PRIOR_INTERACTION","CALL","WHATSAPP","SMS","FIELD_VISIT"],"payments":[7052,4280,2815,2181,1206],"recovery":[526347700,322162900,211839000,163524200,91710090]})
latest_touch["recovery_share_pct"]=(latest_touch.recovery/latest_touch.recovery.sum()*100).round(2)
display(latest_touch)


,latest_touch,payments,recovery,recovery_share_pct
0,NO_PRIOR_INTERACTION,7052,526347700,40.01
1,CALL,4280,322162900,24.49
2,WHATSAPP,2815,211839000,16.10
3,SMS,2181,163524200,12.43
4,FIELD_VISIT,1206,91710090,6.97


### Attribution finding

In the 30-day view, **68.07%** of successful payment value had at least one prior call, but only **24.49%** had a call as the latest interaction. WhatsApp was associated with **56.94%** of recovery value under an any-prior-touch definition, but only **16.10%** under latest-touch.

About **40.01%** of successful payment value had no recorded interaction in the preceding 30 days. I therefore keep channel attribution descriptive.


## 11. PTP and agent productivity

For PTP, I use `KEPT / (KEPT + BROKEN)` and do not automatically treat open promises as failures.

For agents, I use recovery per logged agent-hour as an operational measure. It should not be interpreted as a causal measure of agent effectiveness.


In [23]:

# 15. PTP ANALYSIS + 16. AGENT PRODUCTIVITY


#  PTP ANALYSIS 

ptp = pd.read_csv(RAW / "promises_to_pay.csv")

print("PTP records:", len(ptp))
print("\nPTP status distribution:")
display(ptp["status"].value_counts())

# Calculate matured PTP kept rate
kept = int((ptp["status"] == "KEPT").sum())
broken = int((ptp["status"] == "BROKEN").sum())
open_ptp = int((ptp["status"] == "OPEN").sum())
cancelled = int((ptp["status"] == "CANCELLED").sum())

if kept + broken > 0:
    matured_kept_rate = 100 * kept / (kept + broken)
else:
    matured_kept_rate = np.nan

print("KEPT PTPs:", kept)
print("BROKEN PTPs:", broken)
print("OPEN PTPs:", open_ptp)
print("CANCELLED PTPs:", cancelled)
print("Matured kept rate:", round(matured_kept_rate, 2), "%")


# AGENT PRODUCTIVITY

print("\n" + "=" * 50)
print("AGENT PRODUCTIVITY")
print("=" * 50)

# Load without assuming column names
sessions = pd.read_csv(RAW / "agent_sessions.csv")

print("\nAgent session columns:")
print(sessions.columns.tolist())


# Find possible start/end columns automatically
possible_start_cols = []
possible_end_cols = []

for col in sessions.columns:
    col_lower = col.lower().strip()

    if any(word in col_lower for word in [
        "start", "started", "begin", "beginning", "login", "check_in", "checkin"
    ]):
        possible_start_cols.append(col)

    if any(word in col_lower for word in [
        "end", "ended", "finish", "finishing", "logout", "check_out", "checkout"
    ]):
        possible_end_cols.append(col)

print("\nPossible start columns:", possible_start_cols)
print("Possible end columns:", possible_end_cols)


# Select columns
start_col = possible_start_cols[0] if possible_start_cols else None
end_col = possible_end_cols[0] if possible_end_cols else None


# If automatic detection fails, stop and show the available columns
if start_col is None or end_col is None:
    raise ValueError(
        "\nCould not automatically identify the session start/end columns.\n"
        f"Available columns: {sessions.columns.tolist()}\n"
        f"Possible start columns found: {possible_start_cols}\n"
        f"Possible end columns found: {possible_end_cols}"
    )


print("\nUsing start column:", start_col)
print("Using end column:", end_col)


# Convert timestamps
sessions[start_col] = pd.to_datetime(
    sessions[start_col],
    errors="coerce"
)

sessions[end_col] = pd.to_datetime(
    sessions[end_col],
    errors="coerce"
)


# Calculate duration in hours
sessions["session_hours"] = (
    sessions[end_col] - sessions[start_col]
).dt.total_seconds() / 3600


# Keep valid positive sessions
valid_sessions = sessions[
    sessions["session_hours"].notna() &
    (sessions["session_hours"] > 0)
].copy()


# Summary
print("\nTotal session records:", len(sessions))
print("Valid session records:", len(valid_sessions))
print(
    "Invalid / missing duration records:",
    len(sessions) - len(valid_sessions)
)

print("\nSession length summary:")
print(
    "Mean session hours:",
    round(valid_sessions["session_hours"].mean(), 2)
)

print(
    "Median session hours:",
    round(valid_sessions["session_hours"].median(), 2)
)

print(
    "Maximum session hours:",
    round(valid_sessions["session_hours"].max(), 2)
)

PTP records: 18000

PTP status distribution:


status
BROKEN       4553
CANCELLED    4543
KEPT         4489
OPEN         4415
Name: count, dtype: int64

KEPT PTPs: 4489
BROKEN PTPs: 4553
OPEN PTPs: 4415
CANCELLED PTPs: 4543
Matured kept rate: 49.65 %

AGENT PRODUCTIVITY

Agent session columns:
['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']

Possible start columns: ['login_at']
Possible end columns: ['logout_at']

Using start column: login_at
Using end column: logout_at

Total session records: 15000
Valid session records: 15000
Invalid / missing duration records: 0

Session length summary:
Mean session hours: 5.26
Median session hours: 5.24
Maximum session hours: 10.0


## 12. Counterfactual: what would recovery have looked like without the targeting change?

I would use a treatment group exposed to the new targeting approach and a comparable control group remaining under the old approach.

I would balance the groups on pre-treatment DPD, risk, loan type, outstanding amount, prior collection activity, historical recovery and geography/borrower attributes where available. A Difference-in-Differences design is preferred if pre-treatment trends support it.

The current data is observational, so I do **not** treat the current analysis as proof that targeting caused the recovery movement.


## 13. Investment scenarios

My recommended area to test is **better borrower targeting**. The numbers below are decision scenarios, not observed causal results.


In [24]:
scenario=pd.DataFrame({"scenario":["Downside","Base","Upside"],"uplift_pct":[2,5,10],"incremental_recovery_cr":[4.349055,10.872637,21.745273],"net_value_cr":[-5.650945,0.872637,11.745273],"roi_pct":[-56.51,8.73,117.45]})
display(scenario)
print("Approximate break-even uplift: 4.6%")


,scenario,uplift_pct,incremental_recovery_cr,net_value_cr,roi_pct
0,Downside,2,4.349055,-5.650945,-56.51
1,Base,5,10.872637,0.872637,8.73
2,Upside,10,21.745273,11.745273,117.45


Approximate break-even uplift: 4.6%


# Final conclusion

**What is true:** gross recovery increased 11.03% from February to March, while recovery per eligible account fell 18.37%; payment duplicates materially changed recovery; portfolio mix was broadly stable; and channel attribution changes with the attribution rule.

**What is not proven:** that a specific channel or the targeting strategy caused the recovery improvement, or that the full ₹10 Cr will generate a positive ROI.

**My decision:** run a controlled pilot for **better borrower targeting**, measure incremental recovery against a comparable control group, and scale only if the economics are positive.

> The 11% number is real as a gross movement, but it is not enough by itself to justify a ₹10 Cr investment.
